In [1]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# DuckDB 연결
db_path = '/home/oracle/Coding/wsl_projects/fire_birds/miniprj/data-pipeline/data/duckdb/mimic_total.duckdb'
con = duckdb.connect(db_path)

print("=== 슬라이딩 윈도우 코호트 생성 시작 ===\n")

# Step 1: 기본 코호트 (원래 정의 그대로)
print("Step 1: 기본 포함 기준 적용")
base_cohort_query = """
CREATE OR REPLACE TABLE cohort_base AS
SELECT 
    i.subject_id,
    i.hadm_id,
    i.stay_id,
    CAST(i.intime AS TIMESTAMP) as intime,
    CAST(i.outtime AS TIMESTAMP) as outtime,
    CAST(i.los AS DOUBLE) as los,
    i.first_careunit,
    i.last_careunit,
    CAST(p.anchor_age AS INTEGER) as anchor_age,
    p.gender,
    CAST(p.dod AS TIMESTAMP) as dod,
    CAST(a.admittime AS TIMESTAMP) as admittime,
    CAST(a.dischtime AS TIMESTAMP) as dischtime,
    CAST(a.deathtime AS TIMESTAMP) as deathtime,
    a.hospital_expire_flag,
    -- 첫 번째 입실 여부 확인
    ROW_NUMBER() OVER (PARTITION BY i.subject_id ORDER BY i.intime) as icu_seq
FROM icustays i
INNER JOIN patients p ON i.subject_id = p.subject_id
INNER JOIN admissions a ON i.hadm_id = a.hadm_id
WHERE 
    -- 성인 환자 (18세 이상)
    CAST(p.anchor_age AS INTEGER) >= 18
    -- ICU 체류 24시간 이상
    AND CAST(i.los AS DOUBLE) >= 1.0
"""

con.execute(base_cohort_query)

# 첫 번째 입실만
con.execute("""
CREATE OR REPLACE TABLE cohort_first_stay AS
SELECT * FROM cohort_base
WHERE icu_seq = 1
""")

result = con.execute("SELECT COUNT(*) as count FROM cohort_first_stay").df()
print(f"기본 포함 기준 후: {result['count'][0]:,}명\n")

# Step 2: DNR 환자 제외
print("Step 2: DNR 환자 제외")

dnr_query = """
CREATE OR REPLACE TABLE dnr_patients AS
SELECT DISTINCT c.stay_id
FROM cohort_first_stay c
INNER JOIN chartevents ce ON c.stay_id = ce.stay_id
WHERE 
    ce.itemid = '223758'
    AND CAST(ce.charttime AS TIMESTAMP) <= c.intime + INTERVAL '6 hours'
"""

con.execute(dnr_query)

con.execute("""
CREATE OR REPLACE TABLE cohort_no_dnr AS
SELECT c.*
FROM cohort_first_stay c
WHERE c.stay_id NOT IN (SELECT stay_id FROM dnr_patients)
""")

result = con.execute("SELECT COUNT(*) as count FROM cohort_no_dnr").df()
print(f"DNR 제외 후: {result['count'][0]:,}명\n")

# Step 3: 조기 이벤트 발생자 제외
print("Step 3: 조기 이벤트 발생자 제외 (7시간 이내)")

early_event_query = """
CREATE OR REPLACE TABLE cohort_no_early_event AS
SELECT *
FROM cohort_no_dnr
WHERE 
    outtime > intime + INTERVAL '7 hours'
    AND (deathtime IS NULL OR deathtime > intime + INTERVAL '7 hours')
"""

con.execute(early_event_query)

result = con.execute("SELECT COUNT(*) as count FROM cohort_no_early_event").df()
print(f"조기 이벤트 제외 후: {result['count'][0]:,}명\n")

# Step 4: 필수 활력징후 기록 확인
print("Step 4: 필수 활력징후 기록 확인 (0-6시간)")

vital_signs_query = """
CREATE OR REPLACE TABLE cohort_with_vitals AS
SELECT c.*
FROM cohort_no_early_event c
WHERE EXISTS (
    SELECT 1
    FROM chartevents ce
    WHERE ce.stay_id = c.stay_id
    AND ce.itemid IN (
        '220045', '220210',
        '220050', '220051', '220052',
        '220179', '220180', '220181'
    )
    AND CAST(ce.charttime AS TIMESTAMP) >= c.intime
    AND CAST(ce.charttime AS TIMESTAMP) <= c.intime + INTERVAL '6 hours'
)
"""

con.execute(vital_signs_query)

result = con.execute("SELECT COUNT(*) as count FROM cohort_with_vitals").df()
print(f"필수 활력징후 기록 있는 환자: {result['count'][0]:,}명\n")

# Step 5: 중재 시작 시점 식별
print("Step 5: 중재 시작 시점 식별")

vent_query = """
CREATE OR REPLACE TABLE vent_start_times AS
SELECT 
    stay_id,
    MIN(CAST(starttime AS TIMESTAMP)) as vent_start
FROM procedureevents
WHERE 
    itemid = '225792'
    AND starttime IS NOT NULL
GROUP BY stay_id
"""

con.execute(vent_query)

pressor_query = """
CREATE OR REPLACE TABLE pressor_start_times AS
SELECT 
    stay_id,
    MIN(CAST(starttime AS TIMESTAMP)) as pressor_start
FROM inputevents
WHERE 
    itemid IN ('221906', '221289', '222315', '221662')
    AND starttime IS NOT NULL
    AND rate IS NOT NULL
    AND CAST(rate AS DOUBLE) > 0
GROUP BY stay_id
"""

con.execute(pressor_query)

print("중재 시점 식별 완료\n")

# ========== 핵심: 슬라이딩 윈도우 시점 추가 ==========
print("Step 6: 슬라이딩 윈도우 시점 정보 추가")

sliding_window_query = """
CREATE OR REPLACE TABLE cohort_sliding_window AS
WITH base_cohort AS (
    SELECT 
        c.*,
        v.vent_start,
        p.pressor_start,
        
        -- ICU mortality
        CASE 
            WHEN c.deathtime IS NOT NULL 
            AND c.deathtime <= c.outtime
            THEN 1 ELSE 0
        END as icu_mortality,
        
        -- Hospital mortality
        CASE 
            WHEN c.hospital_expire_flag = '1' THEN 1
            ELSE 0
        END as hospital_mortality
        
    FROM cohort_with_vitals c
    LEFT JOIN vent_start_times v ON c.stay_id = v.stay_id
    LEFT JOIN pressor_start_times p ON c.stay_id = p.stay_id
),

-- 관찰 가능한 시점들 생성 (6시간 간격)
time_windows AS (
    SELECT UNNEST([6, 12, 18, 24, 36, 48]) as observation_hour
)

-- 각 시점별로 행 생성
SELECT 
    bc.*,
    tw.observation_hour,
    bc.intime + (tw.observation_hour || ' hours')::INTERVAL as observation_end_time,
    
    -- 이 시점에서의 예측 윈도우 (observation_hour ~ observation_hour + prediction_horizon)
    -- 예측 구간: +6시간
    CASE 
        WHEN bc.deathtime IS NOT NULL
        AND bc.deathtime > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
        AND bc.deathtime <= bc.intime + ((tw.observation_hour + 6) || ' hours')::INTERVAL
        THEN 1 ELSE 0
    END as death_next_6h,
    
    CASE 
        WHEN bc.vent_start IS NOT NULL
        AND bc.vent_start > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
        AND bc.vent_start <= bc.intime + ((tw.observation_hour + 6) || ' hours')::INTERVAL
        THEN 1 ELSE 0
    END as vent_start_next_6h,
    
    CASE 
        WHEN bc.pressor_start IS NOT NULL
        AND bc.pressor_start > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
        AND bc.pressor_start <= bc.intime + ((tw.observation_hour + 6) || ' hours')::INTERVAL
        THEN 1 ELSE 0
    END as pressor_start_next_6h,
    
    -- 예측 구간: +12시간
    CASE 
        WHEN bc.deathtime IS NOT NULL
        AND bc.deathtime > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
        AND bc.deathtime <= bc.intime + ((tw.observation_hour + 12) || ' hours')::INTERVAL
        THEN 1 ELSE 0
    END as death_next_12h,
    
    CASE 
        WHEN bc.vent_start IS NOT NULL
        AND bc.vent_start > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
        AND bc.vent_start <= bc.intime + ((tw.observation_hour + 12) || ' hours')::INTERVAL
        THEN 1 ELSE 0
    END as vent_start_next_12h,
    
    CASE 
        WHEN bc.pressor_start IS NOT NULL
        AND bc.pressor_start > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
        AND bc.pressor_start <= bc.intime + ((tw.observation_hour + 12) || ' hours')::INTERVAL
        THEN 1 ELSE 0
    END as pressor_start_next_12h,
    
    -- 예측 구간: +24시간
    CASE 
        WHEN bc.deathtime IS NOT NULL
        AND bc.deathtime > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
        AND bc.deathtime <= bc.intime + ((tw.observation_hour + 24) || ' hours')::INTERVAL
        THEN 1 ELSE 0
    END as death_next_24h,
    
    CASE 
        WHEN bc.vent_start IS NOT NULL
        AND bc.vent_start > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
        AND bc.vent_start <= bc.intime + ((tw.observation_hour + 24) || ' hours')::INTERVAL
        THEN 1 ELSE 0
    END as vent_start_next_24h,
    
    CASE 
        WHEN bc.pressor_start IS NOT NULL
        AND bc.pressor_start > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
        AND bc.pressor_start <= bc.intime + ((tw.observation_hour + 24) || ' hours')::INTERVAL
        THEN 1 ELSE 0
    END as pressor_start_next_24h,
    
    -- 통합 라벨들
    CASE 
        WHEN (
            (bc.deathtime IS NOT NULL
             AND bc.deathtime > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
             AND bc.deathtime <= bc.intime + ((tw.observation_hour + 6) || ' hours')::INTERVAL)
            OR
            (bc.vent_start IS NOT NULL
             AND bc.vent_start > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
             AND bc.vent_start <= bc.intime + ((tw.observation_hour + 6) || ' hours')::INTERVAL)
            OR
            (bc.pressor_start IS NOT NULL
             AND bc.pressor_start > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
             AND bc.pressor_start <= bc.intime + ((tw.observation_hour + 6) || ' hours')::INTERVAL)
        ) THEN 1 ELSE 0
    END as composite_next_6h,
    
    CASE 
        WHEN (
            (bc.deathtime IS NOT NULL
             AND bc.deathtime > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
             AND bc.deathtime <= bc.intime + ((tw.observation_hour + 12) || ' hours')::INTERVAL)
            OR
            (bc.vent_start IS NOT NULL
             AND bc.vent_start > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
             AND bc.vent_start <= bc.intime + ((tw.observation_hour + 12) || ' hours')::INTERVAL)
            OR
            (bc.pressor_start IS NOT NULL
             AND bc.pressor_start > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
             AND bc.pressor_start <= bc.intime + ((tw.observation_hour + 12) || ' hours')::INTERVAL)
        ) THEN 1 ELSE 0
    END as composite_next_12h,
    
    CASE 
        WHEN (
            (bc.deathtime IS NOT NULL
             AND bc.deathtime > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
             AND bc.deathtime <= bc.intime + ((tw.observation_hour + 24) || ' hours')::INTERVAL)
            OR
            (bc.vent_start IS NOT NULL
             AND bc.vent_start > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
             AND bc.vent_start <= bc.intime + ((tw.observation_hour + 24) || ' hours')::INTERVAL)
            OR
            (bc.pressor_start IS NOT NULL
             AND bc.pressor_start > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
             AND bc.pressor_start <= bc.intime + ((tw.observation_hour + 24) || ' hours')::INTERVAL)
        ) THEN 1 ELSE 0
    END as composite_next_24h

FROM base_cohort bc
CROSS JOIN time_windows tw
WHERE 
    -- 해당 시점에 여전히 ICU에 있는 환자만
    bc.outtime > bc.intime + (tw.observation_hour || ' hours')::INTERVAL
    -- 해당 시점 이후로 최소 1시간은 더 관찰 가능
    AND bc.outtime >= bc.intime + ((tw.observation_hour + 1) || ' hours')::INTERVAL
"""

con.execute(sliding_window_query)

# 결과 확인
print("\n=== 슬라이딩 윈도우 코호트 통계 ===")
stats = con.execute("""
    SELECT 
        observation_hour,
        COUNT(*) as n_samples,
        SUM(death_next_6h) as deaths_6h,
        SUM(death_next_12h) as deaths_12h,
        SUM(death_next_24h) as deaths_24h,
        SUM(vent_start_next_6h) as vent_6h,
        SUM(pressor_start_next_6h) as pressor_6h,
        SUM(composite_next_6h) as composite_6h,
        SUM(composite_next_12h) as composite_12h,
        SUM(composite_next_24h) as composite_24h,
        COUNT(DISTINCT stay_id) as unique_patients
    FROM cohort_sliding_window
    GROUP BY observation_hour
    ORDER BY observation_hour
""").df()

print(stats)

# 각 시점별 비율
print("\n=== 각 시점별 Positive 비율 ===")
for _, row in stats.iterrows():
    hour = int(row['observation_hour'])
    n = row['n_samples']
    print(f"\n[{hour}시간 시점]")
    print(f"  총 샘플: {n:,}개 (고유 환자: {row['unique_patients']:,}명)")
    print(f"  향후 6h 사망: {row['deaths_6h']} ({row['deaths_6h']/n*100:.2f}%)")
    print(f"  향후 6h 통합: {row['composite_6h']} ({row['composite_6h']/n*100:.2f}%)")
    print(f"  향후 12h 통합: {row['composite_12h']} ({row['composite_12h']/n*100:.2f}%)")
    print(f"  향후 24h 통합: {row['composite_24h']} ({row['composite_24h']/n*100:.2f}%)")

# 샘플 데이터 확인
print("\n=== 샘플 데이터 (동일 환자의 여러 시점) ===")
sample = con.execute("""
    SELECT 
        stay_id,
        observation_hour,
        death_next_6h,
        death_next_12h,
        death_next_24h,
        composite_next_6h,
        composite_next_12h,
        composite_next_24h
    FROM cohort_sliding_window
    WHERE stay_id = (SELECT stay_id FROM cohort_sliding_window LIMIT 1)
    ORDER BY observation_hour
""").df()
print(sample)

print("\n=== 슬라이딩 윈도우 코호트 생성 완료 ===")
print(f"테이블명: cohort_sliding_window")

=== 슬라이딩 윈도우 코호트 생성 시작 ===

Step 1: 기본 포함 기준 적용
기본 포함 기준 후: 54,551명

Step 2: DNR 환자 제외
DNR 제외 후: 37,139명

Step 3: 조기 이벤트 발생자 제외 (7시간 이내)
조기 이벤트 제외 후: 37,134명

Step 4: 필수 활력징후 기록 확인 (0-6시간)
필수 활력징후 기록 있는 환자: 36,400명

Step 5: 중재 시작 시점 식별
중재 시점 식별 완료

Step 6: 슬라이딩 윈도우 시점 정보 추가

=== 슬라이딩 윈도우 코호트 통계 ===
   observation_hour  n_samples  deaths_6h  deaths_12h  deaths_24h  vent_6h  \
0                 6      36400        8.0        29.0       295.0    543.0   
1                12      36400       21.0       115.0       451.0    380.0   
2                18      36400       94.0       266.0       612.0    292.0   
3                24      35555      170.0       333.0       660.0    207.0   
4                36      28026      169.0       304.0       531.0    125.0   
5                48      21860      112.0       205.0       387.0     75.0   

   pressor_6h  composite_6h  composite_12h  composite_24h  unique_patients  
0       594.0        1036.0         1705.0         2650.0            36400  

In [4]:
# ========== 기존 코드 이어서 ==========

import os
from datetime import datetime

# 저장 경로 설정
output_dir = '/home/oracle/Coding/wsl_projects/fire_birds/miniprj/data-pipeline/data/processed'
os.makedirs(output_dir, exist_ok=True)

print("\n" + "="*60)
print("CSV 파일 및 문서 생성 시작")
print("="*60 + "\n")

# 1. 메인 코호트 테이블 저장
print("1. 메인 코호트 테이블 저장 중...")
output_path = os.path.join(output_dir, 'cohort_sliding_window.csv')

con.execute(f"""
    COPY cohort_sliding_window 
    TO '{output_path}' 
    (HEADER, DELIMITER ',')
""")

# 파일 정보
file_size = os.path.getsize(output_path) / (1024 * 1024)
row_count = con.execute("SELECT COUNT(*) as count FROM cohort_sliding_window").fetchone()[0]
unique_patients = con.execute("SELECT COUNT(DISTINCT stay_id) as count FROM cohort_sliding_window").fetchone()[0]
col_count = len(con.execute("DESCRIBE cohort_sliding_window").df())

print(f"✓ 저장 완료: cohort_sliding_window.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 총 행 수: {row_count:,}개")
print(f"  - 고유 환자: {unique_patients:,}명")
print(f"  - 환자당 평균 시점: {row_count/unique_patients:.1f}개")
print(f"  - 컬럼 수: {col_count}개\n")




CSV 파일 및 문서 생성 시작

1. 메인 코호트 테이블 저장 중...
✓ 저장 완료: cohort_sliding_window.csv
  - 파일 크기: 51.75 MB
  - 총 행 수: 194,641개
  - 고유 환자: 36,400명
  - 환자당 평균 시점: 5.3개
  - 컬럼 수: 34개



In [6]:
# ========== 기존 코드 이어서 ==========

import os
from datetime import datetime

# 저장 경로 설정
output_dir = '/home/oracle/Coding/wsl_projects/fire_birds/miniprj/data-pipeline/data/processed'
os.makedirs(output_dir, exist_ok=True)

print("\n" + "="*60)
print("CSV 파일 및 문서 생성 시작")
print("="*60 + "\n")

# 1. 메인 코호트 테이블 저장
print("1. 메인 코호트 테이블 저장 중...")
output_path = os.path.join(output_dir, 'cohort_sliding_window.csv')

con.execute(f"""
    COPY cohort_sliding_window 
    TO '{output_path}' 
    (HEADER, DELIMITER ',')
""")

# 파일 정보
file_size = os.path.getsize(output_path) / (1024 * 1024)
row_count = con.execute("SELECT COUNT(*) as count FROM cohort_sliding_window").fetchone()[0]
unique_patients = con.execute("SELECT COUNT(DISTINCT stay_id) as count FROM cohort_sliding_window").fetchone()[0]
col_count = len(con.execute("DESCRIBE cohort_sliding_window").df())

print(f"✓ 저장 완료: cohort_sliding_window.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 총 행 수: {row_count:,}개")
print(f"  - 고유 환자: {unique_patients:,}명")
print(f"  - 환자당 평균 시점: {row_count/unique_patients:.1f}개")
print(f"  - 컬럼 수: {col_count}개\n")

# 2. 통계 요약 저장
print("2. 통계 요약 저장 중...")
summary_path = os.path.join(output_dir, 'cohort_summary_statistics.csv')

summary_df = con.execute("""
    SELECT 
        observation_hour,
        COUNT(*) as n_samples,
        COUNT(DISTINCT stay_id) as unique_patients,
        SUM(death_next_6h) as deaths_6h,
        SUM(death_next_12h) as deaths_12h,
        SUM(death_next_24h) as deaths_24h,
        SUM(vent_start_next_6h) as vent_6h,
        SUM(vent_start_next_12h) as vent_12h,
        SUM(vent_start_next_24h) as vent_24h,
        SUM(pressor_start_next_6h) as pressor_6h,
        SUM(pressor_start_next_12h) as pressor_12h,
        SUM(pressor_start_next_24h) as pressor_24h,
        SUM(composite_next_6h) as composite_6h,
        SUM(composite_next_12h) as composite_12h,
        SUM(composite_next_24h) as composite_24h,
        ROUND(CAST(SUM(death_next_6h) AS DOUBLE) / COUNT(*) * 100, 2) as death_6h_pct,
        ROUND(CAST(SUM(composite_next_6h) AS DOUBLE) / COUNT(*) * 100, 2) as composite_6h_pct,
        ROUND(CAST(SUM(composite_next_12h) AS DOUBLE) / COUNT(*) * 100, 2) as composite_12h_pct,
        ROUND(CAST(SUM(composite_next_24h) AS DOUBLE) / COUNT(*) * 100, 2) as composite_24h_pct
    FROM cohort_sliding_window
    GROUP BY observation_hour
    ORDER BY observation_hour
""").df()

summary_df.to_csv(summary_path, index=False)
print(f"✓ 저장 완료: cohort_summary_statistics.csv\n")

# 3. 컬럼 정보 저장
print("3. 컬럼 정보 저장 중...")
columns_path = os.path.join(output_dir, 'cohort_columns_info.csv')

columns_df = con.execute("DESCRIBE cohort_sliding_window").df()
columns_df.to_csv(columns_path, index=False)
print(f"✓ 저장 완료: cohort_columns_info.csv\n")


CSV 파일 및 문서 생성 시작

1. 메인 코호트 테이블 저장 중...
✓ 저장 완료: cohort_sliding_window.csv
  - 파일 크기: 51.75 MB
  - 총 행 수: 194,641개
  - 고유 환자: 36,400명
  - 환자당 평균 시점: 5.3개
  - 컬럼 수: 34개

2. 통계 요약 저장 중...
✓ 저장 완료: cohort_summary_statistics.csv

3. 컬럼 정보 저장 중...
✓ 저장 완료: cohort_columns_info.csv



In [9]:
import os
os.chdir("/home/oracle/Coding/wsl_projects/fire_birds/miniprj/data-pipeline/data/processed")
cohort = pd.read_csv('cohort_sliding_window.csv')


In [10]:
cohort.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'los',
       'first_careunit', 'last_careunit', 'anchor_age', 'gender', 'dod',
       'admittime', 'dischtime', 'deathtime', 'hospital_expire_flag',
       'icu_seq', 'vent_start', 'pressor_start', 'icu_mortality',
       'hospital_mortality', 'observation_hour', 'observation_end_time',
       'death_next_6h', 'vent_start_next_6h', 'pressor_start_next_6h',
       'death_next_12h', 'vent_start_next_12h', 'pressor_start_next_12h',
       'death_next_24h', 'vent_start_next_24h', 'pressor_start_next_24h',
       'composite_next_6h', 'composite_next_12h', 'composite_next_24h'],
      dtype='object')